In [ ]:
%cd drive/MyDrive/Thesis

In [ ]:
!nvidia-smi

In [ ]:
import numpy as np
import torch
import gzip
import gc

f = gzip.GzipFile("/content/drive/MyDrive/Thesis/Thesis_Data/cullpdb+profile_5926_filtered.npy.gz", "r")
f5926 = np.load(f)

numproteins = f5926.shape[0]
f5926 =  np.reshape(f5926, (numproteins, 700, 57))

np.shape(f5926)
train_set      = f5926[0:5430,:,:]
test_set       = f5926[5435:5689,:,:]
validation_set = f5926[5690:5926,:,:]

feature_index  =  range(35,57)
target_index =  range(22,31)
del f5926
gc.collect()

In [ ]:
if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

train_x = torch.from_numpy(train_set[:,:,feature_index]).to(dtype=torch.float32, device=device)
print(train_x.size())
train_y = torch.from_numpy(train_set[:,:,target_index])
train_y = torch.argmax(train_y, dim = 1)
train_y.to(dtype=torch.int8, device = device)
print(train_y.size())

traindataset = torch.TensorDataset(train_x, train_y)
loader = torch.DataLoader(dataset,batch_size=128, shuffle=True, drop_last=False)

In [ ]:
class myRNN(torch.nn.Module):
    def __init__(self, input_dim = 22, hidden_dim=256, num_classes = 9, ):
        super(myRNN, self).__init__()
        self.input_size = input_dim
        self.hidden_size = hidden_dim
        self.hidden = None
        self.fc = torch.nn.Linear(in_features=2 * self.datapoint_size + 1, out_features=self.input_size)
        self.gru = torch.nn.GRU(self.input_size, self.hidden_size, num_layers=2)
        

    def reset_hidden(self):
        self.hidden = None
        
    def forward(self, X):
        batch, seq_len, i_shape = X.shape

        X = self.fc(X).view(seq_len, batch, self.input_size)

        X, self.hidden = self.gru(X, self.hidden)
        X = X.view(seq_len, 1, 64, 64)
        X = self.conv1(X)
        X = self.up1(X)
        X = self.conv2(X)
        X = self.up2(X)
        return X



In [ ]:
model = RNN(512, 256).to(device)
model.to(device)

epochs =10
for epoch in range(epochs):
  